# Remove Redundant Comparison Links
Identifies and removes unnecessary directed comparison links from the database.

## Redundancy Rule
A directed link `u → v` (u beat v) is **unnecessary** if a longer directed chain `u → ... → v` (length ≥2) exists. The direct link is redundant because the longer chain already establishes u is better than v.

## Examples
1. Chain: `a → b → c → d → e`, extra link `a → d` → **Redundant** (path `a→b→c→d` exists)
2. Edges: `a→b, b→c, d→c (d beat c), a→d` → **Not redundant** (no longer directed path from a to d)

In [ ]:
import os
import sys
from pathlib import Path

# Add plugin root to path
notebook_path = Path(os.getcwd()).resolve()
plugin_root = notebook_path.parents[1]
print(f"Plugin root: {plugin_root}")

if str(plugin_root) not in sys.path:
    sys.path.insert(0, str(plugin_root))
if str(notebook_path) not in sys.path:
    sys.path.insert(0, str(notebook_path))

print(f"Current working directory: {os.getcwd()}")


In [ ]:
from tqdm.notebook import tqdm

# Import CrystalGraph singleton (already builds graph on import)
from shared.graph.crystal_graph import crystal_graph

# Import delete function
from database.comparisons_table import delete_comparison

print("CrystalGraph loaded. Graph built at:", crystal_graph._chain._built_at)


In [ ]:
# --- CONFIGURATION ---
DRY_RUN = False  # Set to False to apply actual changes
DELETE_COMPARISONS = True  # Set to True to delete redundant links from DB
# Both DRY_RUN=False and DELETE_COMPARISONS=True required to make changes
# ---------------------


In [ ]:
print("Using prebuilt CrystalGraph from shared module...")
print(f"Total images: {len(crystal_graph._images)}")
print(f"Total comparisons: {len(crystal_graph._comparisons)}")
print(
    f"Directed edges (winner->loser): {sum(len(v) for v in crystal_graph._chain._worse_than.values())}"
)


In [ ]:
from collections import defaultdict

# Force rebuild graph from current DB to avoid stale singleton data
crystal_graph.rebuild_from_database()

print("Identifying redundant edges...")
redundant_edges = set()
necessary_edges = []
self_links = []
double_link_removals = []
transitive_removals = []

# 1. Detect self-links (image compared to itself)
for u, losers in crystal_graph._chain._worse_than.items():
    for v in losers:
        if u == v:
            self_links.append((u, v))
            redundant_edges.add((u, v))

# 2. Detect redundant directed edges via transitive chains
for u, losers in crystal_graph._chain._worse_than.items():
    for v in losers:
        if u != v and crystal_graph.is_redundant(u, v):
            transitive_removals.append((u, v))
            redundant_edges.add((u, v))

# 3. Detect double links (a->b and b->a) - keep the newest
pair_comparisons = defaultdict(list)
for comp in crystal_graph._comparisons:
    key = (comp["filename_a"], comp["filename_b"])
    pair_comparisons[key].append(comp)

for (ca, cb), comps in pair_comparisons.items():
    winners = set(c["winner"] for c in comps)
    if ca in winners and cb in winners:
        edge_ab = (ca, cb)
        edge_ba = (cb, ca)
        if edge_ab in redundant_edges or edge_ba in redundant_edges:
            continue
        ts_a = max(c["timestamp"] for c in comps if c["winner"] == ca)
        ts_b = max(c["timestamp"] for c in comps if c["winner"] == cb)
        if ts_a >= ts_b:
            edge_to_remove = edge_ba
        else:
            edge_to_remove = edge_ab
        double_link_removals.append(edge_to_remove)
        redundant_edges.add(edge_to_remove)

# Build necessary_edges from remaining edges
for u, losers in crystal_graph._chain._worse_than.items():
    for v in losers:
        if (u, v) not in redundant_edges:
            necessary_edges.append((u, v))

print(f"\nSelf-links found: {len(self_links)}")
print(f"Transitively redundant edges: {len(transitive_removals)}")
print(f"Double-link removals (kept newest): {len(double_link_removals)}")
print(f"Total edges to remove: {len(redundant_edges)}")
print(f"Edges to keep: {len(necessary_edges)}")
print(f"Total directed edges: {len(redundant_edges) + len(necessary_edges)}")


In [ ]:
redundant_list = list(redundant_edges)
if redundant_list:
    print("\nExample redundant edges (u -> v):")
    for u, v in redundant_list[:5]:
        print(f"  {u} -> {v}")
else:
    print("\nNo redundant edges found.")

if necessary_edges:
    print("\nExample necessary edges (u -> v):")
    for u, v in necessary_edges[:5]:
        print(f"  {u} -> {v}")


In [ ]:
if DELETE_COMPARISONS and not DRY_RUN:
    print(f"\nDeleting {len(redundant_edges)} redundant edges from database...")
    deleted = 0
    errors = 0
    for u, v in tqdm(redundant_edges, desc="Deleting"):
        # u is winner, v is loser; delete_comparison handles canonicalization
        result = delete_comparison(u, v, u)
        if result > 0:
            deleted += 1
        else:
            errors += 1
    print(f"Deleted {deleted} comparison records. Errors: {errors}")
elif DRY_RUN and DELETE_COMPARISONS:
    print(f"\n[DRY RUN] Would delete {len(redundant_edges)} redundant edges.")
else:
    print("\nNo changes made (DELETE_COMPARISONS=False or DRY_RUN=True).")


In [ ]:
# Recalculate comparison_count for all affected images after deletion
from database.images_table import update_image_confidence, get_image, get_all_images
from database.comparisons_table import get_comparison_count

if DELETE_COMPARISONS and not DRY_RUN and deleted > 0:
    print(f"\nRecalculating comparison_count for affected images...")

    # Collect all unique filenames from deleted edges
    affected_files = set()
    for u, v in redundant_edges:
        affected_files.add(u)
        affected_files.add(v)

    updated = 0
    for filename in tqdm(affected_files, desc="Updating counts"):
        actual_count: int = get_comparison_count(filename)
        img = get_image(filename)
        if img and img.get("comparison_count", 0) != actual_count:
            update_image_confidence(filename, img["confidence"], actual_count)
            updated += 1
    print(f"Updated comparison_count for {updated} images")

    # Also sync ALL images to match actual DB counts (for consistency)
    # print("\nSyncing comparison_count for ALL images...")
    # all_images = get_all_images()
    # fixed = 0
    # for img in tqdm(all_images, desc="Syncing all"):
    #     filename = img["filename"]
    #     actual = get_comparison_count(filename)
    #     if img.get("comparison_count", 0) != actual:
    #         update_image_confidence(filename, img["confidence"], actual)
    #         fixed += 1
    # print(f"Fixed comparison_count for {fixed}/{len(all_images)} total images")
elif DELETE_COMPARISONS and not DRY_RUN:
    print("\nNo redundant edges deleted, skipping count update.")


In [ ]:
from collections import defaultdict

# Rebuild crystal_graph from DB to verify clean state
print("Verifying clean state after deletions...")
crystal_graph.rebuild_from_database()

# Check for self-links
remaining_self_links = [
    (u, v) for u, losers in crystal_graph._chain._worse_than.items() for v in losers if u == v
]
if remaining_self_links:
    print(f"WARNING: {len(remaining_self_links)} self-links still remain!")
    for u, v in remaining_self_links[:5]:
        print(f"  {u} -> {v}")
else:
    print("✓ No self-links remain.")

# Check for double links
pair_comparisons = defaultdict(list)
for comp in crystal_graph._comparisons:
    key = (comp["filename_a"], comp["filename_b"])
    pair_comparisons[key].append(comp)
remaining_double = [
    (ca, cb)
    for (ca, cb), comps in pair_comparisons.items()
    if ca in set(c["winner"] for c in comps) and cb in set(c["winner"] for c in comps)
]
if remaining_double:
    print(f"WARNING: {len(remaining_double)} contradictory pairs still remain!")
    for ca, cb in remaining_double:
        comps = pair_comparisons[(ca, cb)]
        print(f"  ({ca}, {cb})")
        for c in comps:
            print(f"    winner={c['winner']} ts={c['timestamp']}")
else:
    print("✓ No contradictory double-links remain.")

print(
    f"\nFinal state: {len(crystal_graph._comparisons)} comparisons, {len(crystal_graph._images)} images"
)
print(f"Final directed edges: {sum(len(v) for v in crystal_graph._chain._worse_than.values())}")
